# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cokezero20/FlyRank_AI_ML_Internship_NATIVIDAD/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib

In [2]:
from datasets import load_dataset
from google.colab import userdata
import pandas as pd
import numpy as np
from datetime import date

# ============================================================================
# STEP 1: LOAD DATA (Jan-Apr 2026)
# ============================================================================
HF_TOKEN = userdata.get('HF_Token')

dataset = load_dataset(
    'FlyRank/internship-warehouse',
    name='fact_content_daily_performance',
    token=HF_TOKEN,
    streaming=True
)

train_split = dataset['train']

print("Loading January-April 2026 data...")
data_rows = []
batch_size = 100000
batch_count = 0

for batch in train_split.iter(batch_size=batch_size):
    batch_count += 1
    batch_df = pd.DataFrame(batch)

    # Filter for Jan-Apr 2026 and ga4_data_available = True
    if isinstance(batch_df['report_date'].iloc[0], str):
        batch_df['report_date'] = pd.to_datetime(batch_df['report_date']).dt.date

    data_batch = batch_df[
        (batch_df['report_date'] >= date(2026, 1, 1)) &
        (batch_df['report_date'] <= date(2026, 4, 30)) &
        (batch_df['ga4_data_available'] == True)
    ]

    if len(data_batch) > 0:
        data_rows.append(data_batch)
        print(f"  Batch {batch_count}: {len(data_batch):,} rows")

df_raw = pd.concat(data_rows, ignore_index=True)
print(f"✓ Total loaded: {len(df_raw):,} rows\n")


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading January-April 2026 data...
  Batch 200: 1,046 rows
  Batch 201: 702 rows
  Batch 202: 1,998 rows
  Batch 203: 1,008 rows
  Batch 204: 2,250 rows
  Batch 205: 1,597 rows
  Batch 206: 3,844 rows
  Batch 207: 351 rows
  Batch 208: 2,704 rows
  Batch 209: 1,323 rows
  Batch 210: 2,163 rows
  Batch 211: 1,730 rows
  Batch 212: 1,371 rows
  Batch 213: 1,721 rows
  Batch 214: 1,673 rows
  Batch 215: 594 rows
  Batch 216: 357 rows
  Batch 217: 1,035 rows
  Batch 218: 775 rows
  Batch 219: 1 rows
  Batch 220: 2,154 rows
  Batch 221: 59 rows
  Batch 222: 3,539 rows
  Batch 223: 2,723 rows
  Batch 224: 378 rows
  Batch 225: 896 rows
  Batch 226: 3,179 rows
  Batch 227: 1,314 rows
  Batch 228: 555 rows
  Batch 229: 2,693 rows
  Batch 230: 683 rows
  Batch 231: 2,031 rows
  Batch 232: 1,499 rows
  Batch 233: 1,270 rows
  Batch 234: 10 rows
  Batch 235: 500 rows
  Batch 236: 2,793 rows
  Batch 237: 931 rows
  Batch 238: 2,195 rows
  Batch 239: 1,294 rows
  Batch 240: 1,721 rows
  Batch 241: 

In [4]:
print("Preparing data for feature engineering...")

# Convert dates to datetime
df_raw['report_date'] = pd.to_datetime(df_raw['report_date'])

# FILTER TO TRAINING DATA FIRST (Jan-Mar only, drop Apr to save RAM)
df_raw = df_raw[df_raw['report_date'].dt.month.isin([1, 2, 3])].copy()
df_raw = df_raw.sort_values(['content_hash_id', 'report_date']).reset_index(drop=True)

print(f"Training data only: {len(df_raw):,} rows\n")

Preparing data for feature engineering...
Training data only: 675,073 rows



In [5]:
print("Computing rolling window features (March only, 30d window)...")

core_metrics = ['gsc_impressions', 'gsc_clicks', 'ga4_pageviews']
windows = [30]

# Keep only March
df_march = df_raw[df_raw['report_date'].dt.month == 3].copy()
unique_contents = df_march['content_hash_id'].unique()

features_list = []

for i, content_id in enumerate(unique_contents):
    if (i + 1) % 25000 == 0:
        print(f"  {i + 1:,} / {len(unique_contents):,}")

    group = df_march[df_march['content_hash_id'] == content_id].sort_values('report_date').reset_index(drop=True)

    feature_df = group[['content_hash_id', 'client_hash_id', 'report_date']].copy()

    for metric in core_metrics:
        feature_df[f'{metric}_prev_30d'] = group[metric].rolling(window=30, min_periods=1).sum().values

    features_list.append(feature_df)

df_features = pd.concat(features_list, ignore_index=True)
print(f"✓ Done: {df_features.shape}")

Computing rolling window features (March only, 30d window)...
  25,000 / 90,489
  50,000 / 90,489
  75,000 / 90,489
✓ Done: (413966, 6)


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.